# Tokenizers for the GWDG open-weight models

This notebook exercises **all** of the tokenizer features in `saia_python`
(the `[tokenizer]` extra). **No SAIA API key is needed** — tokenizers come from
Hugging Face, not the chat API.

The install cell below sets up the optional `[tokenizer]` extra.

Covered:

1. Optional authenticated Hugging Face downloads (`HF_TOKEN`)
2. The model → Hugging Face repo registry and `resolve_repo` / `repo_url`
3. Downloading tokenizer files (never weights) to `~/saia_python/tokenizers/`
4. Loading a tokenizer
5. `chat_template_tokens` — length, special/structural **overhead** (absolute **and**
   relative), and **subword fertility**
6. System prompt from a `.md` file
7. `gpt-oss-120b` — counting the tokens of a system prompt
8. `qwen3-embedding-4b` (the ARCANA RAG embedding model) — sizing a chunk
9. **Token distribution over a directory** of RAG files/chunks (recursive, incl. images)
10. Tolerance, the convenience wrappers, `tiktoken`, and batch download


## Install the `[tokenizer]` extra

Idempotent: it only runs `pip` when the dependencies are missing, so re-running it is a no-op and an editable / development install is left untouched.

```bash
pip install saia-python[tokenizer]
```

In [1]:
# Idempotent install of the optional [tokenizer] extra.
import importlib.util
import subprocess
import sys

_missing = [
    name
    for name in ("transformers", "huggingface_hub", "tiktoken")
    if importlib.util.find_spec(name) is None
]
if _missing:
    print("Installing saia-python[tokenizer] (missing:", ", ".join(_missing) + ") ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "saia-python[tokenizer]"],
        check=True,
    )
else:
    print("saia-python[tokenizer] already available - nothing to install.")

saia-python[tokenizer] already available - nothing to install.


## 0. Setup

In [2]:
import os

import saia_python as s
from saia_python import tokenizer as tk

os.environ.setdefault("HF_HUB_DISABLE_TELEMETRY", "1")
print("saia_python", s.__version__)

saia_python 0.5.0


## 1. Authenticated Hugging Face downloads (optional)

Public tokenizers download anonymously, but a Hugging Face token lifts the Hub's
anonymous rate limit (the *"sending unauthenticated requests to the HF Hub"*
warning) and is required for gated repos. `saia_python` resolves a token from, in
order: the `HF_TOKEN` / `HUGGING_FACE_HUB_TOKEN` environment variable, then a
`.env` or `.saia_env` file in the working directory, then your home directory —
the same idea as the SAIA API key.

**First-time setup**

1. Create a *read* token at <https://huggingface.co/settings/tokens>.
2. Add it next to your SAIA key, in `.env` or `.saia_env`:

   ```
   HF_TOKEN=hf_xxxxxxxxxxxxxxxxxxxxxxxx
   ```

It is then picked up automatically by every download below.

In [ ]:
hf_token = tk.load_hf_token()
print(
    "HF token configured:",
    hf_token is not None,
    "(None is fine - public repos download anonymously, just rate-limited)",
)

## 2. The model → repository registry

`GET /models` does not say where each model's tokenizer lives, so `saia_python`
ships the mapping from the GWDG catalogue (open-weight chat models **and** the
embedding models such as `qwen3-embedding-4b`). `resolve_repo` accepts a model
id, a catalogue display name, or a full `org/name` repo.

In [ ]:
print(f"{len(tk.GWDG_MODEL_REPOS)} models with a known tokenizer repo\n")
for mid in (
    "openai-gpt-oss-120b",
    "qwen3-coder-30b-a3b-instruct",
    "qwen3-embedding-4b",
):
    print(f"  {mid:<30} -> {tk.GWDG_MODEL_REPOS[mid]}")

print("\nresolve_repo accepts id / display name / full repo:")
print("  by id          :", tk.resolve_repo("openai-gpt-oss-120b"))
print("  by display name:", tk.resolve_repo("GPT OSS 120B"))
print("  passthrough    :", tk.resolve_repo("some-org/Custom-Tokenizer"))
print("\nrepo_url:", tk.repo_url("qwen3-embedding-4b"))

## 3. Download a tokenizer (files only, never the weights)

Defaults to `~/saia_python/tokenizers/<org>/<name>/`. Override per call with
`cache_dir=...` or globally with `SAIA_TOKENIZER_DIR`. We use an ungated Qwen
model, so it works without a token.

In [ ]:
MODEL = "qwen3-coder-30b-a3b-instruct"

print("default cache dir:", tk.tokenizer_dir())
path = tk.download_tokenizer(MODEL)  # tokenizer files only
print("downloaded to    :", path)

files = sorted(p.name for p in path.iterdir())
print("files            :", files)
biggest = max((p.stat().st_size for p in path.iterdir()), default=0)
print(f"largest file     : {biggest / 1e6:.2f} MB  (no multi-GB weights)")

## 4. Load the tokenizer

In [ ]:
tok = tk.load_tokenizer(MODEL)
print("loaded     :", type(tok).__name__)
print("vocab size :", tok.vocab_size)

## 5. Chat-template tokens — overhead (absolute **and** relative) & fertility

`chat_template_tokens` returns a `ChatTokenCount` with the templated length, the
raw-text length, the special/structural-token **overhead** (now also as ratios:
relative to the text, and as a share of the full prompt), and the **subword
fertility** with and without special tokens.

In [ ]:
r = tk.chat_template_tokens(
    MODEL,
    system="You are a terse assistant.",
    user="Write a haiku about linseed oil and mussel shells.",
    add_generation_prompt=True,
)
print("templated tokens   :", r.num_tokens)
print("raw text tokens    :", r.num_text_tokens)
print("special overhead   :", r.overhead_tokens)
print(f"   relative to text : {r.overhead_ratio_text:.2f}x the text tokens")
print(f"   share of total   : {r.overhead_ratio_total:.1%} of the prompt")
print("fertility (text)   :", round(r.fertility, 3))
print("fertility (+spec)  :", round(r.fertility_with_special, 3))
print("\nsummary:", r.summary())

## 6. System prompt from a `.txt` / `.md` file

In [ ]:
import tempfile
from pathlib import Path

tmp = Path(tempfile.mkdtemp())
sys_md = tmp / "system_prompt.md"
sys_md.write_text("# Persona\nYou are a helpful lighthouse keeper.", encoding="utf-8")
# Or point at your own file: sys_md = Path("path/to/system_prompt.md")

r = tk.chat_template_tokens(MODEL, system_file=str(sys_md), user="hi")
print("system prompt read from:", sys_md.name)
print("num_words:", r.num_words, "| num_tokens:", r.num_tokens)

## 7. `gpt-oss-120b` — how many tokens is a system prompt?

OpenAI's open-weight `gpt-oss-120b` uses the *harmony* chat format, whose
scaffolding around a system prompt is substantial — a good illustration of the
overhead accounting.

In [ ]:
import tempfile
from pathlib import Path

GPT_OSS = "openai-gpt-oss-120b"
SYSTEM = (
    "You are GPT-OSS, a careful assistant. Cite sources and refuse unsafe requests."
)

# Inline system prompt:
sys_only = tk.chat_template_tokens(GPT_OSS, system=SYSTEM, add_generation_prompt=False)
print("system prompt, templated tokens :", sys_only.num_tokens)
print("system prompt, raw text tokens  :", sys_only.num_text_tokens)
print(
    "harmony/structural overhead     :",
    sys_only.overhead_tokens,
    f"({sys_only.overhead_ratio_total:.0%} of the templated total)",
)

In [ ]:
# Same system prompt, read from a .txt / .md file:
sys_file = Path(tempfile.mkdtemp()) / "system_prompt.md"
sys_file.write_text(SYSTEM, encoding="utf-8")
# Or point at your own file: sys_file = Path("path/to/system_prompt.md")

from_file = tk.chat_template_tokens(
    GPT_OSS, system_file=str(sys_file), add_generation_prompt=False
)

print("from file:", sys_file.name, "->", from_file.num_tokens, "templated tokens")
print("system prompt, templated tokens :", from_file.num_tokens)
print("system prompt, raw text tokens  :", from_file.num_text_tokens)
print(
    "harmony/structural overhead     :",
    from_file.overhead_tokens,
    f"({from_file.overhead_ratio_total:.0%} of the templated total)",
)

## 8. `qwen3-embedding-4b` — sizing a RAG chunk

This is the embedding model ARCANA's RAG pipeline uses internally. Its tokenizer
sizes the chunks you embed (encoded as raw text, no chat template).

In [ ]:
EMBED = "qwen3-embedding-4b"
etok = tk.load_tokenizer(EMBED)

chunk = """
    ---
    Author: fschwar4
    Title: Title of the file
    Description: 'This is a small description of the file content.'
    Filename: 2026_06_21_example_long_version_max_150_chars.pdf
    Extension: pdf
    Number of Pages: 20
    Version: 1.0
    ---
   """
n_tokens = len(etok.encode(chunk, add_special_tokens=False))
print(f"{EMBED} ({type(etok).__name__}, vocab {etok.vocab_size})")
print(f"chunk of {len(chunk.split())} words -> {n_tokens} tokens")

## 9. Token distribution over a directory of RAG files / chunks

`token_distribution` walks a directory **recursively** (into subdirectories),
tokenizes each text file as raw content, **estimates** a token cost for each
image, and returns per-file rows plus aggregate statistics — built for sizing a
whole RAG corpus against a model's tokenizer (here the embedding model).

We build a small sample corpus (with a subdirectory and an image) to scan.

In [ ]:
import tempfile
from pathlib import Path

corpus = Path(tempfile.mkdtemp()) / "rag_corpus"
(corpus / "docs").mkdir(parents=True)
(corpus / "docs" / "lighthouse.md").write_text(
    "# Lighthouse\nRepainted every seven years with linseed oil and mussel shells. "
    * 6,
    encoding="utf-8",
)
(corpus / "docs" / "maintenance.md").write_text(
    "## Maintenance\nThe lantern room is cleaned every spring. " * 3, encoding="utf-8"
)
(corpus / "intro.txt").write_text("A short introduction chunk.", encoding="utf-8")
(corpus / "data.bin").write_bytes(b"\x00\x01\x02")  # binary -> skipped

# Optional image, to show images are included in the distribution.
try:
    from PIL import Image

    Image.new("RGB", (640, 480), (90, 90, 90)).save(corpus / "docs" / "diagram.png")
except Exception:
    pass

# Or point at your own RAG directory: corpus = Path("path/to/markdown")

dist = tk.token_distribution(corpus, EMBED)  # recurse, incl. images
print(dist.summary())

In [ ]:
print("by kind     :", dist.by_kind())
print("by extension:", dist.by_extension())
print(
    "stats       :",
    {k: (round(v, 1) if isinstance(v, float) else v) for k, v in dist.stats().items()},
)

print("\nper-file:")
for f in dist.files:
    note = f" ({f.note})" if f.note else ""
    print(f"  {f.kind:8} {f.num_tokens:>6} tok  {f.path}{note}")

print("\ntoken-count histogram (per file):")
for lo, hi, c in dist.histogram(5):
    print(f"  {lo:6.0f}-{hi:<6.0f} | {chr(9608) * c} ({c})")

## 10. Tolerance — missing / empty parts never raise

In [ ]:
only_system = tk.chat_template_tokens(MODEL, system="only a system prompt")
print(
    "system-only -> tokens:",
    only_system.num_tokens,
    "| warnings:",
    only_system.warnings,
)

empty = tk.chat_template_tokens(MODEL, messages=[])
print(
    "empty       -> tokens:", empty.num_tokens, "| warning count:", len(empty.warnings)
)

print("\nNeither call raised an exception.")

## 11. Convenience wrappers

In [ ]:
print("chat_template_length  :", tk.chat_template_length(MODEL, user="hello world"))
print("special_token_overhead:", tk.special_token_overhead(MODEL, user="hello world"))

sentence = "The quick brown fox jumps over the lazy dog"
print("\nsubword fertility of a 9-word sentence:")
print("  include_special=False:", round(tk.subword_fertility(MODEL, user=sentence), 3))
print(
    "  include_special=True :",
    round(tk.subword_fertility(MODEL, user=sentence, include_special=True), 3),
)

## 12. `tiktoken` — externally hosted OpenAI models

GPT-5.x / o3 / … are relayed to OpenAI and have no downloadable Hugging Face tokenizer; their byte-pair encoding is available through `tiktoken`.

In [ ]:
print(
    "o200k tokens for 'hello world':",
    tk.count_tiktoken_tokens("hello world", model="gpt-5"),
)
print(
    "encodings used by the external models:",
    sorted(set(tk.OPENAI_TIKTOKEN_ENCODINGS.values())),
)

## 13. Download many tokenizers at once

`download_all_tokenizers` loops the download and tolerates per-model failures (a gated repo or network error becomes `None`, the batch continues). Restricted to already-cached models here to keep it fast.

In [ ]:
result = tk.download_all_tokenizers(models=[MODEL, EMBED], verbose=True)
print("\nresult:", {k: str(v) for k, v in result.items()})

# In practice, drive it from the live available-models list:
#   client = s.SAIAClient()
#   client.tokenizers.download_all()        # every open-weight model

---

That covers every public entry point of the `[tokenizer]` feature:
`load_hf_token` · `GWDG_MODEL_REPOS` · `resolve_repo` · `repo_url` ·
`available_open_models` · `download_tokenizer` · `download_all_tokenizers` ·
`load_tokenizer` · `chat_template_tokens` (with relative overhead) ·
`chat_template_length` · `special_token_overhead` · `subword_fertility` ·
`count_tiktoken_tokens` · `token_distribution` · `client.tokenizers`.